07/11/2026
Carga de la capa Golden

Se inicia con una validación de conteos y cuadre de valores contra silver y seguidamente con la carga de informacion a las tablas:
1. kpi_facturacion_periodo
2. kpi_recaudo_periodo
3. kpi_cartera_suscriptor
4. kpi_indice_recaudo
5. kpi_calidad_datos
6. kpi_detalle_suscriptor (cartera + consumo m3 y nombre del cliente por suscriptor y periodo)

Cada tabla se sobrescribe completa en cada ejecución (`insertInto(overwrite=True)`), así los periodos con pagos tardíos se actualizan sin duplicar registros.

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_pysmartdata")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
if "catalogo" not in dbutils.widgets.getAll(): dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo")
if "esquema_source" not in dbutils.widgets.getAll(): dbutils.widgets.text("esquema_source", "")
esquema_source = dbutils.widgets.get("esquema_source")
if "esquema_sink" not in dbutils.widgets.getAll(): dbutils.widgets.text("esquema_sink", "")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_facturacion_transformed = spark.table(f"{catalogo}.{esquema_source}.facturacion_transformed")
df_recaudos_transformed = spark.table(f"{catalogo}.{esquema_source}.recaudos_transformed")

In [0]:
# Conciliación: bronze = silver válidos + silver rechazados
conteos = []
for tabla in ["facturacion", "recaudos"]:
    bronze = spark.table(f"{catalogo}.bronze.{tabla}").count()
    validos = spark.table(f"{catalogo}.{esquema_source}.{tabla}_transformed").count()
    rechazados = spark.table(f"{catalogo}.{esquema_source}.{tabla}_rechazados").count()
    conteos.append((tabla, bronze, validos, rechazados, bronze - validos - rechazados))

df_conteos = spark.createDataFrame(conteos, "tabla string, bronze long, silver_validos long, silver_rechazados long, diferencia long")
display(df_conteos)

# Conteos por periodo en silver
display(
    df_facturacion_transformed.groupBy("periodo").agg(F.count("*").alias("registros_facturacion"), F.countDistinct("suscriptor").alias("suscriptores_facturados"))
    .join(df_recaudos_transformed.groupBy("periodo").agg(F.count("*").alias("registros_recaudos"), F.countDistinct("suscriptor").alias("suscriptores_con_pago")), "periodo", "full")
    .orderBy("periodo")
)

In [0]:
def cargar_golden(df, tabla):
    """Sobrescribe la tabla golden respetando el orden de columnas de la tabla destino."""
    destino = f"{catalogo}.{esquema_sink}.{tabla}"
    columnas = spark.table(destino).columns
    df.withColumn("fecha_carga", F.current_timestamp()).select(*columnas).write.insertInto(destino, overwrite=True)
    print(f"{destino}: {spark.table(destino).count():,} registros cargados")

In [0]:
df_kpi_facturacion = (
    df_facturacion_transformed
    .groupBy("periodo", "estrato", "codigo_concepto", "concepto")
    .agg(
        F.countDistinct("suscriptor").alias("num_suscriptores"),
        F.sum("consumo_m3").cast("bigint").alias("consumo_total_m3"),
        F.round(F.sum("valor_cargo_fijo"), 2).alias("valor_cargo_fijo"),
        F.round(F.sum("valor_consumo"), 2).alias("valor_consumo"),
        F.round(F.sum("valor_total"), 2).alias("valor_total_facturado"),
    )
    .withColumn("consumo_promedio_m3", F.round(F.col("consumo_total_m3") / F.col("num_suscriptores"), 2))
    .withColumn("tarifa_media_m3", F.when(F.col("consumo_total_m3") > 0, F.round(F.col("valor_consumo") / F.col("consumo_total_m3"), 2)))
)

cargar_golden(df_kpi_facturacion, "kpi_facturacion_periodo")
display(spark.table(f"{catalogo}.{esquema_sink}.kpi_facturacion_periodo").orderBy("periodo", "estrato").limit(20))

In [0]:
# Periodos presentes en facturación: pagos de esos periodos son recaudo CORRIENTE; los demás son recuperación de CARTERA
df_periodos_facturados = df_facturacion_transformed.select("periodo").distinct().withColumn("es_facturado", F.lit(True))

df_kpi_recaudo = (
    df_recaudos_transformed
    .join(F.broadcast(df_periodos_facturados), "periodo", "left")
    .withColumn("tipo_recaudo", F.when(F.col("es_facturado"), "CORRIENTE").otherwise("CARTERA"))
    .withColumnRenamed("descripcion", "descripcion_entidad")
    .groupBy("periodo", "tipo_recaudo", "entidad", "descripcion_entidad", "codigo_punto_recaudo", "descripcion_punto_recaudo", "fecha_pago")
    .agg(
        F.count("*").alias("num_pagos"),
        F.countDistinct("suscriptor").alias("num_suscriptores"),
        F.round(F.sum("valor_pagado"), 2).alias("valor_recaudado"),
    )
    .withColumn("pago_promedio", F.round(F.col("valor_recaudado") / F.col("num_pagos"), 2))
)

cargar_golden(df_kpi_recaudo, "kpi_recaudo_periodo")
display(
    spark.table(f"{catalogo}.{esquema_sink}.kpi_recaudo_periodo")
    .groupBy("tipo_recaudo").agg(F.sum("num_pagos").alias("num_pagos"), F.round(F.sum("valor_recaudado"), 2).alias("valor_recaudado"))
)

In [0]:
from pyspark.sql import Window

TOLERANCIA = 1.0  # diferencia en pesos que se considera pago completo (redondeos)
periodo_min_facturado = df_facturacion_transformed.agg(F.min("periodo")).first()[0]

# Facturado por suscriptor y periodo
df_facturado = (
    df_facturacion_transformed.groupBy("suscriptor", "periodo")
    .agg(F.max("estrato").alias("estrato_fact"), F.sum(F.coalesce("valor_total", F.lit(0.0))).alias("valor_facturado"))
)

# Pagado por suscriptor y periodo
df_pagado = (
    df_recaudos_transformed.groupBy("suscriptor", "periodo")
    .agg(F.sum("valor_pagado").alias("valor_pagado"), F.count("*").alias("num_pagos"), F.max("fecha_pago").alias("fecha_ultimo_pago"))
)

# Estrato más reciente del suscriptor, para completar filas de cartera antigua sin factura
w = Window.partitionBy("suscriptor").orderBy(F.col("periodo").desc())
df_estrato = (
    df_facturacion_transformed.filter(F.col("estrato").isNotNull())
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("suscriptor", F.col("estrato").alias("estrato_actual"))
)

df_kpi_cartera = (
    df_facturado.join(df_pagado, ["suscriptor", "periodo"], "full")
    .join(df_estrato, "suscriptor", "left")
    .withColumn("tiene_factura", F.col("valor_facturado").isNotNull())
    .withColumn("estrato", F.coalesce("estrato_fact", "estrato_actual"))
    .withColumn("valor_facturado", F.round(F.coalesce("valor_facturado", F.lit(0.0)), 2))
    .withColumn("valor_pagado", F.round(F.coalesce("valor_pagado", F.lit(0.0)), 2))
    .withColumn("num_pagos", F.coalesce("num_pagos", F.lit(0)).cast("bigint"))
    .withColumn("saldo_pendiente", F.round(F.col("valor_facturado") - F.col("valor_pagado"), 2))
    .withColumn(
        "estado_pago",
        F.when(~F.col("tiene_factura") & (F.col("periodo") < periodo_min_facturado), "RECUPERACION_CARTERA")
         .when(~F.col("tiene_factura"), "PAGO_SIN_FACTURA")
         .when(F.abs(F.col("saldo_pendiente")) <= TOLERANCIA, "AL_DIA")
         .when(F.col("valor_pagado") == 0, "SIN_PAGO")
         .when(F.col("saldo_pendiente") > TOLERANCIA, "PAGO_PARCIAL")
         .otherwise("SALDO_A_FAVOR")
    )
)

cargar_golden(df_kpi_cartera, "kpi_cartera_suscriptor")
display(
    spark.table(f"{catalogo}.{esquema_sink}.kpi_cartera_suscriptor")
    .groupBy("estado_pago").agg(F.count("*").alias("registros"), F.round(F.sum("saldo_pendiente"), 2).alias("saldo_pendiente"))
    .orderBy(F.desc("registros"))
)

In [0]:
# Solo periodos facturados: se excluyen RECUPERACION_CARTERA y PAGO_SIN_FACTURA
df_kpi_indice = (
    spark.table(f"{catalogo}.{esquema_sink}.kpi_cartera_suscriptor")
    .filter(~F.col("estado_pago").isin("RECUPERACION_CARTERA", "PAGO_SIN_FACTURA"))
    .groupBy("periodo", "estrato")
    .agg(
        F.count("*").alias("num_suscriptores_facturados"),
        F.sum(F.when(F.col("valor_pagado") > 0, 1).otherwise(0)).cast("bigint").alias("num_suscriptores_con_pago"),
        F.sum(F.when(F.col("estado_pago").isin("SIN_PAGO", "PAGO_PARCIAL"), 1).otherwise(0)).cast("bigint").alias("num_suscriptores_morosos"),
        F.round(F.sum("valor_facturado"), 2).alias("valor_facturado"),
        F.round(F.sum("valor_pagado"), 2).alias("valor_recaudado"),
        F.round(F.sum("saldo_pendiente"), 2).alias("saldo_pendiente"),
    )
    .withColumn("indice_recaudo", F.when(F.col("valor_facturado") > 0, F.round(F.col("valor_recaudado") / F.col("valor_facturado"), 4)))
    .withColumn("porcentaje_morosidad", F.round(F.col("num_suscriptores_morosos") / F.col("num_suscriptores_facturados"), 4))
)

cargar_golden(df_kpi_indice, "kpi_indice_recaudo")
display(spark.table(f"{catalogo}.{esquema_sink}.kpi_indice_recaudo").orderBy("periodo", "estrato"))

In [0]:
def calidad_por_tabla(tabla):
    df_bronze = spark.table(f"{catalogo}.bronze.{tabla}")
    df_validos = spark.table(f"{catalogo}.{esquema_source}.{tabla}_transformed")
    df_rech = spark.table(f"{catalogo}.{esquema_source}.{tabla}_rechazados")

    # Base por periodo (join null-safe porque los rechazados pueden tener periodo nulo)
    b = df_bronze.groupBy("periodo").agg(F.count("*").alias("registros_bronze"))
    v = df_validos.groupBy(F.col("periodo").alias("p_v")).agg(F.count("*").alias("registros_validos"))
    base = b.join(v, b.periodo.eqNullSafe(v.p_v), "left").drop("p_v")

    # Fila resumen por periodo (motivo nulo = total de registros rechazados)
    r_total = df_rech.groupBy(F.col("periodo").alias("p_r")).agg(F.count("*").alias("registros_rechazados"), F.max("fecha_validacion").alias("fecha_validacion"))
    resumen = (base.join(r_total, base.periodo.eqNullSafe(r_total.p_r), "left").drop("p_r")
               .withColumn("motivo_rechazo", F.lit(None).cast("string")))

    # Filas por motivo (un registro puede tener varios motivos)
    r_motivo = (df_rech.select(F.col("periodo").alias("p_r"), F.explode("motivos_rechazo").alias("motivo_rechazo"), "fecha_validacion")
                .groupBy("p_r", "motivo_rechazo").agg(F.count("*").alias("registros_rechazados"), F.max("fecha_validacion").alias("fecha_validacion")))
    motivos = r_motivo.join(base, base.periodo.eqNullSafe(r_motivo.p_r), "left").withColumn("periodo", F.coalesce("periodo", "p_r")).drop("p_r")

    return (resumen.unionByName(motivos)
            .withColumn("tabla_origen", F.lit(tabla))
            .withColumn("registros_validos", F.coalesce("registros_validos", F.lit(0)).cast("bigint"))
            .withColumn("registros_rechazados", F.coalesce("registros_rechazados", F.lit(0)).cast("bigint"))
            .withColumn("porcentaje_rechazo", F.when(F.col("registros_bronze") > 0, F.round(F.col("registros_rechazados") / F.col("registros_bronze"), 6))))

df_kpi_calidad = calidad_por_tabla("facturacion").unionByName(calidad_por_tabla("recaudos"))

cargar_golden(df_kpi_calidad, "kpi_calidad_datos")
display(
    spark.table(f"{catalogo}.{esquema_sink}.kpi_calidad_datos")
    .filter(F.col("registros_rechazados") > 0)
    .orderBy("tabla_origen", "periodo", F.col("motivo_rechazo").asc_nulls_first())
)

In [0]:
# Detalle por suscriptor y periodo: cartera golden + consumo m3 y nombre del cliente desde silver.
# Fuente de la hoja "Detalle por suscriptor" del dashboard.

# La tabla externa se crea (si no existe) en el mismo contenedor golden de las demas tablas KPI
ruta_golden = spark.sql(f"DESCRIBE DETAIL {catalogo}.{esquema_sink}.kpi_cartera_suscriptor").first()["location"].rsplit("/", 1)[0]

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.{esquema_sink}.kpi_detalle_suscriptor (
  suscriptor INT,
  nombre_cliente STRING,
  periodo INT,
  estrato INT,
  estado_pago STRING,
  consumo_m3 BIGINT,
  valor_facturado DOUBLE,
  valor_pagado DOUBLE,
  saldo_pendiente DOUBLE,
  num_pagos BIGINT,
  fecha_ultimo_pago DATE,
  fecha_carga TIMESTAMP
)
USING DELTA
LOCATION '{ruta_golden}/kpi_detalle_suscriptor'
COMMENT 'Detalle por suscriptor y periodo: facturado, recaudado, saldo, estado de pago y consumo m3'
""")

# Consumo y nombre por suscriptor y periodo (se suman los conceptos, igual que kpi_facturacion_periodo)
df_consumo = (
    df_facturacion_transformed
    .groupBy("suscriptor", "periodo")
    .agg(
        F.sum("consumo_m3").cast("bigint").alias("consumo_m3"),
        F.max(F.trim("nombre_cliente")).alias("nombre_cliente"),
    )
)

# Left join: se conservan todos los registros de cartera, incluidos PAGO_SIN_FACTURA (sin consumo ni nombre)
df_kpi_detalle = (
    spark.table(f"{catalogo}.{esquema_sink}.kpi_cartera_suscriptor").drop("fecha_carga")
    .join(df_consumo, ["suscriptor", "periodo"], "left")
)

cargar_golden(df_kpi_detalle, "kpi_detalle_suscriptor")

In [0]:
# Paso 7 (antes Paso 6): validacion de la carga golden
g = lambda t: spark.table(f"{catalogo}.{esquema_sink}.{t}")
suma = lambda df, c: df.agg(F.round(F.sum(c), 2)).first()[0]

cartera_facturada = g("kpi_cartera_suscriptor").filter(~F.col("estado_pago").isin("RECUPERACION_CARTERA", "PAGO_SIN_FACTURA"))

chequeos = [
    ("Facturado silver vs kpi_facturacion_periodo", suma(df_facturacion_transformed, "valor_total"), suma(g("kpi_facturacion_periodo"), "valor_total_facturado")),
    ("Recaudado silver vs kpi_recaudo_periodo", suma(df_recaudos_transformed, "valor_pagado"), suma(g("kpi_recaudo_periodo"), "valor_recaudado")),
    ("Pagos silver vs kpi_recaudo_periodo", float(df_recaudos_transformed.count()), float(g("kpi_recaudo_periodo").agg(F.sum("num_pagos")).first()[0])),
    ("Facturado silver vs kpi_cartera_suscriptor", suma(df_facturacion_transformed, "valor_total"), suma(g("kpi_cartera_suscriptor"), "valor_facturado")),
    ("Recaudado silver vs kpi_cartera_suscriptor", suma(df_recaudos_transformed, "valor_pagado"), suma(g("kpi_cartera_suscriptor"), "valor_pagado")),
    ("Facturado cartera vs kpi_indice_recaudo", suma(cartera_facturada, "valor_facturado"), suma(g("kpi_indice_recaudo"), "valor_facturado")),
    ("Registros cartera vs kpi_detalle_suscriptor", float(g("kpi_cartera_suscriptor").count()), float(g("kpi_detalle_suscriptor").count())),
    ("Facturado cartera vs kpi_detalle_suscriptor", suma(g("kpi_cartera_suscriptor"), "valor_facturado"), suma(g("kpi_detalle_suscriptor"), "valor_facturado")),
    ("Consumo kpi_facturacion_periodo vs kpi_detalle_suscriptor", suma(g("kpi_facturacion_periodo"), "consumo_total_m3"), suma(g("kpi_detalle_suscriptor"), "consumo_m3")),
    ("Rechazados silver vs kpi_calidad_datos",
     float(spark.table(f"{catalogo}.{esquema_source}.facturacion_rechazados").count() + spark.table(f"{catalogo}.{esquema_source}.recaudos_rechazados").count()),
     float(g("kpi_calidad_datos").filter(F.col("motivo_rechazo").isNull()).agg(F.sum("registros_rechazados")).first()[0])),
]

df_chequeos = (spark.createDataFrame(chequeos, "chequeo string, valor_origen double, valor_golden double")
               .withColumn("diferencia", F.round(F.col("valor_origen") - F.col("valor_golden"), 2))
               .withColumn("resultado", F.when(F.abs(F.col("diferencia")) <= 1, "OK").otherwise("REVISAR")))
display(df_chequeos)